# Solution E2 · TruncatedSVD does not center: the first component is the mean

**Type**: Find the bug

## Objective

Find out why a reduction to 2 dimensions with `TruncatedSVD` on the digits gives worse results than `PCA` (accuracy 0.41 against 0.60) and an "explained variance" that looks small: `TruncatedSVD` **does not center the data**, so its first component is almost the direction of the mean. Fix it by centering first.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
mean_direction = X.mean(0) / np.linalg.norm(X.mean(0))

## Commented solution

In [ ]:
svd_acc = cross_val_score(make_pipeline(TruncatedSVD(2, random_state=0), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
pca_acc = cross_val_score(make_pipeline(PCA(2), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
print(f"accuracy with 2 components: TruncatedSVD {svd_acc:.3f} | PCA {pca_acc:.3f}")

**Where the bug is.** `PCA` centers the data (subtracts the mean of every column) and finds the directions of maximum variance **around the mean**. `TruncatedSVD` does the SVD of the data **as they are**: if all the pixels are positive the direction that explains the most "energy" is simply that
of the mean (the mean image), which does not distinguish the digits. The second component is the true first component of the PCA: a dimension has been lost.

In [ ]:
svd = TruncatedSVD(2, random_state=0).fit(X)
pca = PCA(2).fit(X)
svd_cosine = float(abs(svd.components_[0] @ mean_direction))
pca_cosine = float(abs(pca.components_[0] @ mean_direction))
svd_evr, pca_evr = svd.explained_variance_ratio_.sum(), pca.explained_variance_ratio_.sum()
print(f"cosine between the first component and the direction of the mean: TruncatedSVD {svd_cosine:.4f} | PCA {pca_cosine:.4f}")
print(f"'explained variance' with 2 components: TruncatedSVD {svd_evr:.3f} | PCA {pca_evr:.3f}")

In [ ]:
centered = make_pipeline(StandardScaler(with_std=False), TruncatedSVD(2, algorithm="arpack"), LogisticRegression(max_iter=3000))
centered_svd_acc = cross_val_score(centered, X, y, cv=cv).mean()
print(f"TruncatedSVD after centering: {centered_svd_acc:.3f} (PCA: {pca_acc:.3f})")

## Check

In [ ]:
with verify():
    check(svd_acc < 0.5 and pca_acc > 0.55, "TruncatedSVD below 0.5, PCA above 0.55", "don't modify the colleague's code")
    check(svd_cosine is not None and svd_cosine > 0.99 and pca_cosine < 0.05, "the first component of TruncatedSVD coincides with the direction of the mean (cosine > 0.99); the PCA one is almost orthogonal (< 0.05)", "np.abs(components_[0] @ mean_direction)")
    check(svd_evr is not None and svd_evr < 0.2 and pca_evr > 0.27, "TruncatedSVD 'explains' less than 20%, PCA over 27%: they are not comparable", "sum(explained_variance_ratio_)")
    check(centered_svd_acc is not None and abs(centered_svd_acc - pca_acc) < 0.01, "after centering the data TruncatedSVD and PCA give the same result (within 0.01)", "StandardScaler(with_std=False) before TruncatedSVD")
done()

## What you learned

- **PCA = SVD of the centered data; `TruncatedSVD` = SVD without centering.** They are the same thing only if the data already have mean zero. With nonnegative pixels the first component of `TruncatedSVD` is the mean image (cosine 0.99998) and the two useful axes reduce to one: 0.41 accuracy
  against 0.60.
- The **"explained variance"** of `TruncatedSVD` (0.18 for two components) is not comparable with that of PCA (0.29): it is computed on the variance of the transformed columns relative to the variance of the original data, but without having removed the mean. Don't use that number to choose `k`.
- `TruncatedSVD` exists for a valid reason: on **sparse matrices** (text with TF-IDF, counts) centering would make the matrix dense and unmanageable. In that case you use it (the technique is called LSA) knowing that the first component will contain the "mean" of the corpus.
- Rule: dense data → `PCA`; sparse data → `TruncatedSVD`; if you want PCA on sparse data, center implicitly with other solvers or reduce the dimension first.

## Common pitfall

Choosing `TruncatedSVD` "because it is faster" on dense data. It is not the same transformation: it changes the result, and no error warns you.

## Interview question

<details><summary>What is the difference between PCA and TruncatedSVD?</summary>

PCA centers the data and then does the SVD, so it finds the directions of maximum variance around the mean. TruncatedSVD does the SVD directly on the matrix, without centering: it is suited to sparse matrices (where centering would destroy the sparsity, as in LSA for text), but on dense data with
a nonzero mean the first component aligns with the mean. If the data are already centered they coincide.

</details>